# 1. Data preparation

This notebook prepares the dataset used by every model in the study:

1. downloads EuroSAT through torchvision (about 94 MB, on the first run only);
2. draws the stratified 70 / 15 / 15 split and checks its fingerprint;
3. decodes every image once and computes the normalisation statistics of the training images;
4. shows the split sizes, the class distribution and sample images.

The training notebooks (03, 04 and 05) repeat steps 1 to 3 themselves, so each of them can also run
on its own.

**Writes:** `results/dataset.json`, `results/figures/class_distribution.png`,
`results/figures/sample_grid.png`.

## Setup

Folders, the random seed, the device, and small helpers for files and figures. Every notebook in
this folder starts with the same setup.

In [ ]:
import json
import os
import platform
import random
import re
import subprocess
import sys
from pathlib import Path

import numpy as np
import torch
import torchvision

# Everything is read from and written to folders next to this notebook.
DATA_DIR = Path("data")                # the downloaded dataset and a decoded-pixel cache
RESULTS_DIR = Path("results")          # metrics, figures and tables
CHECKPOINT_DIR = Path("checkpoints")   # trained weights

SEED = 42            # fixes the split, the initial weights, augmentation and batch order
NUM_CLASSES = 10     # EuroSAT land-cover classes
INPUT_SHAPE = (3, 64, 64)

# cuBLAS needs this workspace setting for repeatable matrix products on a GPU.
CUBLAS_WORKSPACE = ":4096:8"


def seed_everything(seed, deterministic=True):
    """Seed Python, NumPy and PyTorch, so a run gives the same numbers every time.

    With ``deterministic=True``, cuDNN uses deterministic algorithms and stops
    auto-tuning, and PyTorch prefers deterministic implementations. This costs a
    little speed. Without it, two runs of the same experiment can differ by a few
    tenths of a percent, the same size as some differences between the models.
    A few GPU operations have no deterministic version; PyTorch then warns
    instead of stopping.
    """
    if deterministic:
        os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", CUBLAS_WORKSPACE)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)  # seeds the CPU and every GPU
    torch.backends.cudnn.deterministic = deterministic
    torch.backends.cudnn.benchmark = not deterministic
    torch.use_deterministic_algorithms(deterministic, warn_only=True)


def cpu_name():
    """The processor's model name, or an empty string if the operating system won't say."""
    name = ""
    try:
        if sys.platform == "win32":
            import winreg

            key = winreg.OpenKey(
                winreg.HKEY_LOCAL_MACHINE, r"HARDWARE\DESCRIPTION\System\CentralProcessor\0"
            )
            name = str(winreg.QueryValueEx(key, "ProcessorNameString")[0])
        elif sys.platform.startswith("linux"):
            with open("/proc/cpuinfo", encoding="utf-8") as handle:
                for line in handle:
                    if line.lower().startswith("model name"):
                        name = line.split(":", 1)[1]
                        break
        elif sys.platform == "darwin":
            name = subprocess.run(
                ["sysctl", "-n", "machdep.cpu.brand_string"],
                capture_output=True, text=True, timeout=5, check=False,
            ).stdout
    except Exception:
        name = ""
    name = name or platform.processor() or ""
    name = re.sub(r"\((R|TM|C)\)", "", name, flags=re.IGNORECASE)  # trademark signs
    return " ".join(name.split())


def describe_device(device):
    """A readable name for a device, such as ``cuda:0 (Tesla T4)``.

    Every timing is recorded together with this name: a time means little
    without the hardware it was measured on.
    """
    device = torch.device(device)
    if device.type == "cuda":
        index = device.index if device.index is not None else torch.cuda.current_device()
        return f"cuda:{index} ({torch.cuda.get_device_name(index)})"
    name = cpu_name()
    return f"cpu ({name})" if name else "cpu"


#: Model name -> function building it; filled in by the model definitions below.
MODEL_BUILDERS = {}


def build_model(name, num_classes=NUM_CLASSES):
    """Seed PyTorch, then build ``name``: the same seed always gives the same initial weights.

    The three optimizer runs rely on this to start from identical weights, so
    that their comparison measures the optimizer and not the initialisation.
    """
    torch.manual_seed(SEED)
    return MODEL_BUILDERS[name](num_classes)


def count_parameters(model, trainable_only=True):
    """Number of parameters; frozen ones are left out when ``trainable_only``."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad or not trainable_only)


def model_size_kb(model):
    """Size of the saved weights (``state_dict``) in KB, where 1 KB = 1,024 bytes.

    Saving the weights, as a device would store them, rather than counting 4
    bytes per parameter, also counts buffers such as batch-norm statistics.
    """
    import io

    buffer = io.BytesIO()
    torch.save(model.state_dict(), buffer)
    return buffer.getbuffer().nbytes / 1024


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Python       {platform.python_version()} ({platform.system()})")
print(f"PyTorch      {torch.__version__}")
print(f"torchvision  {torchvision.__version__}")
print(f"NumPy        {np.__version__}")
print(f"Device       {describe_device(DEVICE)}")

Figures are shown inline at a small size and saved at print quality. Each model and each optimizer
keeps one colour and one name in every figure and table.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.dpi": 100,            # inline figures: small, so the notebook stays light
    "savefig.dpi": 200,           # saved figures: print quality
    "savefig.bbox": "tight",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "legend.frameon": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "lines.linewidth": 1.8,
})

MODEL_LABELS = {
    "model_a": "Model A",
    "model_b": "Model B",
    "mobilenet_v2": "MobileNetV2",
    "squeezenet1_1": "SqueezeNet 1.1",
}
MODEL_COLORS = {
    "model_a": "#4C72B0",
    "model_b": "#DD8452",
    "mobilenet_v2": "#55A868",
    "squeezenet1_1": "#C44E52",
}
OPTIMIZER_LABELS = {"sgd": "SGD", "sgd_momentum": "SGD + momentum", "adam": "Adam"}
OPTIMIZER_COLORS = {"sgd": "#8172B3", "sgd_momentum": "#937860", "adam": "#DA8BC3"}


def save_figure(fig, path):
    """Save a figure as a PNG, creating its folder if needed."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=200, bbox_inches="tight", facecolor="white")

Results are saved as JSON and Markdown, so the tables can be rebuilt without retraining.

In [ ]:
def write_json(path, payload):
    """Write ``payload`` as indented JSON, creating the folder if needed."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8", newline="\n")


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def write_text(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8", newline="\n")


def format_cell(value, spec=None):
    """One table cell: thousands separators for whole numbers, ``spec`` for decimals."""
    if value is None:
        return "-"
    if isinstance(value, bool):
        return str(value)
    if spec is not None and isinstance(value, (int, float)):
        return format(value, spec)
    if isinstance(value, int):
        return f"{value:,}"
    if isinstance(value, float):
        return f"{value:,.1f}" if abs(value) >= 1000 else f"{value:.4g}"
    return str(value).replace("|", "\\|")


def markdown_table(headers, rows, caption=None, formats=None):
    """A Markdown table. Number columns are right-aligned; ``formats`` fixes their decimals."""
    formats = formats or [None] * len(headers)

    def numeric(column):
        cells = [row[column] for row in rows if row[column] is not None]
        return bool(cells) and all(
            isinstance(cell, (int, float)) and not isinstance(cell, bool) for cell in cells
        )

    lines = [f"*{caption}*", ""] if caption else []
    lines.append("| " + " | ".join(headers) + " |")
    lines.append("|" + "|".join("---:" if numeric(c) else "---" for c in range(len(headers))) + "|")
    for row in rows:
        lines.append("| " + " | ".join(format_cell(v, s) for v, s in zip(row, formats)) + " |")
    return "\n".join(lines) + "\n"

## Data

EuroSAT (Helber et al., 2019): 27,000 Sentinel-2 satellite images of 64×64 pixels, in 10 land-cover
classes. It is downloaded through torchvision and split 70 / 15 / 15 into training, validation and
test sets, stratified by class.

The split is drawn from a fixed seed with NumPy's legacy `RandomState`, whose random sequence does not
change between NumPy versions, so every machine draws the same split. A SHA-256 fingerprint of the
split list (each image's path, label and split) is checked against the fingerprint of the split used
for the reported results: if the dataset or the code differed, the check stops the notebook.

In [ ]:
import csv
import hashlib
import io
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass

from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2

SPLIT_NAMES = ("train", "val", "test")
SPLIT_FRACTIONS = {"train": 0.70, "val": 0.15, "test": 0.15}
IMAGE_SIZE = (64, 64)
EXPECTED_IMAGES = 27_000
BATCH_SIZE = 64

# SHA-256 of the split behind every reported result (see split_fingerprint below).
EXPECTED_SPLIT_SHA256 = "9fc7491cc0b8266013ca7f91e4a0988f39464988545251ee1361552823ea162b"


def download_eurosat():
    """Download EuroSAT through torchvision (about 94 MB, only once) and list its images.

    Returns:
        The class names, in label order (torchvision sorts the folder names), and
        one row per image, sorted by its path inside the dataset, such as
        ``AnnualCrop/AnnualCrop_1.jpg``.
    """
    dataset = torchvision.datasets.EuroSAT(root=str(DATA_DIR), download=True)
    classes = list(dataset.classes)
    rows = [
        {
            "relative_path": f"{classes[label]}/{Path(path).name}",
            "label_index": int(label),
            "label_name": classes[label],
            "source": Path(path),
        }
        for path, label in dataset.samples
    ]
    rows.sort(key=lambda row: row["relative_path"])
    if len(classes) != NUM_CLASSES or len(rows) != EXPECTED_IMAGES:
        raise RuntimeError(
            f"found {len(classes)} classes and {len(rows)} images, expected {NUM_CLASSES} and "
            f"{EXPECTED_IMAGES}: the download is incomplete. Delete {DATA_DIR / 'eurosat'} and run again."
        )
    return classes, rows


def split_sizes(n, fractions):
    """Train, validation and test sizes for one class of ``n`` images."""
    sizes = [round(n * fractions["train"]), round(n * fractions["val"])]
    sizes.append(n - sum(sizes))
    if n >= 3:  # move images from train so that no split is empty
        for index in (1, 2):
            if sizes[index] < 1:
                sizes[0] -= 1 - sizes[index]
                sizes[index] = 1
    return sizes


def stratified_split(labels, fractions, seed):
    """Assign every image to ``train``, ``val`` or ``test``, class by class.

    Within each class the images are shuffled, then cut in the given proportions.
    Splitting per class keeps the class balance identical in all three splits.

    Returns:
        The split name of each image, in the order of ``labels``.
    """
    labels = np.asarray(labels)
    assignment = np.empty(len(labels), dtype=object)
    rng = np.random.RandomState(seed)
    for label in np.unique(labels):  # sorted, so the random sequence is fixed
        members = rng.permutation(np.flatnonzero(labels == label))
        start = 0
        for name, count in zip(SPLIT_NAMES, split_sizes(len(members), fractions)):
            assignment[members[start : start + count]] = name
            start += count
    return assignment.tolist()


def split_fingerprint(rows):
    """SHA-256 of the split list, written as a CSV of path, label index, label name and split."""
    buffer = io.StringIO(newline="")
    writer = csv.writer(buffer, lineterminator="\n")
    writer.writerow(["relative_path", "label_index", "label_name", "split"])
    for row in rows:
        writer.writerow([row["relative_path"], row["label_index"], row["label_name"], row["split"]])
    return hashlib.sha256(buffer.getvalue().encode("utf-8")).hexdigest()


def load_pixels(rows, cache_path):
    """Every image as one ``(N, 64, 64, 3)`` ``uint8`` array, in row order.

    Decoding a JPEG is a fixed step with nothing to learn, so it is done once and
    the pixels are cached next to the dataset. Augmentation and normalisation are
    still applied fresh in every epoch, on top of these pixels.
    """
    height, width = IMAGE_SIZE
    if cache_path.exists():
        pixels = np.load(cache_path)
        if pixels.shape == (len(rows), height, width, 3):
            return pixels

    pixels = np.empty((len(rows), height, width, 3), dtype=np.uint8)

    def decode(position):
        with Image.open(rows[position]["source"]) as image:
            array = np.asarray(image.convert("RGB"))
        if array.shape != (height, width, 3):
            raise ValueError(f"{rows[position]['relative_path']} is not {width}x{height}")
        pixels[position] = array

    with ThreadPoolExecutor(max_workers=min(8, os.cpu_count() or 1)) as pool:
        list(pool.map(decode, range(len(rows))))
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    np.save(cache_path, pixels)
    return pixels


def channel_stats(pixels, indices, chunk=1024):
    """Per-channel mean and standard deviation of the selected images, on a 0 to 1 scale.

    Computed in chunks with 64-bit sums, so the whole training set never needs
    more than a few tens of MB of extra memory.
    """
    indices = np.asarray(indices)
    total = np.zeros(3, dtype=np.float64)
    total_sq = np.zeros(3, dtype=np.float64)
    for start in range(0, len(indices), chunk):
        block = pixels[indices[start : start + chunk]].astype(np.float64) / 255.0
        total += block.sum(axis=(0, 1, 2))
        total_sq += np.square(block).sum(axis=(0, 1, 2))
    count = len(indices) * pixels.shape[1] * pixels.shape[2]
    mean = total / count
    std = np.sqrt(np.maximum(total_sq / count - np.square(mean), 0.0))
    return mean.tolist(), std.tolist()


@dataclass
class EuroSATData:
    """The prepared dataset: images, labels, the split and the normalisation statistics."""

    class_names: list
    rows: list
    labels: np.ndarray
    splits: dict          # split name -> indices into ``pixels``
    pixels: np.ndarray
    norm_mean: list       # of the training images only
    norm_std: list
    fingerprint: str


def prepare_data():
    """Download EuroSAT, draw the split, check its fingerprint and decode the images.

    The normalisation statistics come from the training images only: including
    validation or test images would leak held-out information into training.
    """
    classes, rows = download_eurosat()
    labels = [row["label_index"] for row in rows]
    for row, split in zip(rows, stratified_split(labels, SPLIT_FRACTIONS, SEED)):
        row["split"] = split

    fingerprint = split_fingerprint(rows)
    if fingerprint != EXPECTED_SPLIT_SHA256:
        raise RuntimeError(
            f"the split's fingerprint is {fingerprint[:16]}, not {EXPECTED_SPLIT_SHA256[:16]}: "
            "this is not the split behind the reported results"
        )

    pixels = load_pixels(rows, DATA_DIR / f"eurosat_pixels_{fingerprint[:16]}.npy")
    split_of = np.array([row["split"] for row in rows])
    splits = {name: np.flatnonzero(split_of == name) for name in SPLIT_NAMES}
    mean, std = channel_stats(pixels, splits["train"])
    return EuroSATData(
        class_names=classes,
        rows=rows,
        labels=np.array(labels, dtype=np.int64),
        splits=splits,
        pixels=pixels,
        norm_mean=mean,
        norm_std=std,
        fingerprint=fingerprint,
    )

Training images get random flips and quarter turns in every epoch. Satellite images have no natural
"up", so a flipped or rotated image is still a valid example. Validation and test images are never
augmented, so they are identical every time a model is evaluated.

In [ ]:
class RandomRot90:
    """Rotate an image by 0, 90, 180 or 270 degrees, chosen at random.

    A quarter turn of a square image loses no pixels, unlike an arbitrary angle.
    """

    def __call__(self, image):
        quarter_turns = int(torch.randint(0, 4, (1,)).item())
        return torch.rot90(image, quarter_turns, dims=(-2, -1))

    def __repr__(self):
        return "RandomRot90()"


def build_transform(split, mean, std):
    """Augmentation for the training split only, then scaling to 0 to 1 and normalisation."""
    steps = []
    if split == "train":
        steps += [v2.RandomHorizontalFlip(p=0.5), v2.RandomVerticalFlip(p=0.5), RandomRot90()]
    steps += [v2.ToDtype(torch.float32, scale=True), v2.Normalize(mean=list(mean), std=list(std))]
    return v2.Compose(steps)


class PixelDataset(Dataset):
    """Images from the decoded-pixel array, returned as ``(normalised tensor, label)``."""

    def __init__(self, pixels, indices, labels, transform):
        self.pixels = pixels
        self.indices = np.asarray(indices, dtype=np.int64)
        self.labels = np.asarray(labels, dtype=np.int64)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        image = torch.from_numpy(self.pixels[self.indices[item]]).permute(2, 0, 1)  # HWC to CHW
        return self.transform(image), int(self.labels[item])


def make_loaders(data, mean, std):
    """Training, validation and test DataLoaders, normalised with ``mean`` and ``std``.

    The training loader reshuffles in every epoch, with its own generator seeded
    from ``SEED``; validation and test keep a fixed order.
    """
    loaders = {}
    for split in SPLIT_NAMES:
        indices = data.splits[split]
        dataset = PixelDataset(
            data.pixels, indices, data.labels[indices], build_transform(split, mean, std)
        )
        is_train = split == "train"
        loaders[split] = DataLoader(
            dataset,
            batch_size=BATCH_SIZE,
            shuffle=is_train,
            generator=torch.Generator().manual_seed(SEED) if is_train else None,
            num_workers=0,
            pin_memory=torch.cuda.is_available(),
            drop_last=False,
        )
    return loaders

## Looking at the split

In [ ]:
import pandas as pd

SPLIT_COLORS = {"train": "#2F5E8E", "val": "#6C9BCF", "test": "#B3CDE8"}


def split_counts(data):
    """Images per class in each split, with totals."""
    table = pd.DataFrame(
        [[int(np.sum(data.labels[data.splits[split]] == label)) for split in SPLIT_NAMES]
         for label in range(len(data.class_names))],
        index=pd.Index(data.class_names, name="class"),
        columns=list(SPLIT_NAMES),
    )
    table.loc["Total"] = [len(data.splits[split]) for split in SPLIT_NAMES]
    table["total"] = table[list(SPLIT_NAMES)].sum(axis=1)
    return table.astype(int)


def plot_class_distribution(data):
    """Each class's share of every split; equal bars mean stratification kept the class balance."""
    counts = split_counts(data)
    positions = np.arange(len(data.class_names))
    width = 0.8 / len(SPLIT_NAMES)
    fig, ax = plt.subplots(figsize=(max(6.0, 0.7 * len(data.class_names) + 1.5), 4.2),
                           layout="constrained")
    tallest = 0.0
    for offset, split in enumerate(SPLIT_NAMES):
        total = int(counts.loc["Total", split])
        shares = [100 * int(counts.loc[name, split]) / total for name in data.class_names]
        tallest = max(tallest, *shares)
        ax.bar(positions + (offset - (len(SPLIT_NAMES) - 1) / 2) * width, shares, width,
               label=f"{split} ({total:,} images)", color=SPLIT_COLORS[split])
    ax.set_xticks(positions, data.class_names, rotation=35, ha="right")
    ax.set_ylabel("share of the split (%)")
    ax.set_ylim(0, tallest * 1.3)  # room for the legend above the bars
    ax.set_title("Class distribution in each split")
    ax.grid(axis="x", visible=False)
    ax.legend(loc="upper center", ncols=len(SPLIT_NAMES))
    save_figure(fig, RESULTS_DIR / "figures" / "class_distribution.png")
    plt.show()


def plot_sample_grid(data, per_class=4):
    """Random training images, one row per class, unaugmented, as the models receive them."""
    rng = np.random.RandomState(SEED)
    names = data.class_names
    fig, axes = plt.subplots(len(names), per_class, squeeze=False,
                             figsize=(0.95 * per_class + 1.9, 0.95 * len(names) + 0.5),
                             layout="constrained")
    for label, name in enumerate(names):
        train = [position for position, row in enumerate(data.rows)
                 if row["split"] == "train" and row["label_index"] == label]
        picks = rng.choice(len(train), size=min(per_class, len(train)), replace=False)
        for column, ax in enumerate(axes[label]):
            ax.set_xticks([])
            ax.set_yticks([])
            ax.grid(False)
            for spine in ax.spines.values():
                spine.set_visible(False)
            if column < len(picks):
                ax.imshow(data.pixels[train[picks[column]]], interpolation="nearest")
        axes[label][0].set_ylabel(name, rotation=0, ha="right", va="center", labelpad=6)
    fig.suptitle("Training images at 64×64, as the models receive them")
    save_figure(fig, RESULTS_DIR / "figures" / "sample_grid.png")
    plt.show()

## Prepare the dataset

The first run downloads the dataset and decodes all 27,000 images, which takes a minute or two;
later runs reuse both.

In [ ]:
data = prepare_data()
print(f"{len(data.rows):,} images in {len(data.class_names)} classes: {', '.join(data.class_names)}")
print(f"Split fingerprint (SHA-256): {data.fingerprint}")
print("It matches the split behind the reported results.")

## Split sizes

The totals split 70 / 15 / 15, and every class is split in the same proportions.

In [ ]:
split_counts(data)

## Class distribution

In [ ]:
plot_class_distribution(data)

## Sample images

In [ ]:
plot_sample_grid(data)

## Normalisation statistics

Per-channel mean and standard deviation of the training images, on a 0 to 1 scale. The custom
models are normalised with these; the pretrained models with the ImageNet statistics their weights
were trained with.

In [ ]:
write_json(RESULTS_DIR / "dataset.json", {
    "num_images": len(data.rows),
    "class_names": data.class_names,
    "split_counts": {split: int(len(data.splits[split])) for split in SPLIT_NAMES},
    "split_sha256": data.fingerprint,
    "norm_mean": data.norm_mean,
    "norm_std": data.norm_std,
})
print("mean", [round(value, 4) for value in data.norm_mean])
print("std ", [round(value, 4) for value in data.norm_std])